# AOM Double-Pass MATLAB Script Reproduction

This notebook reproduces `files/AOM_model_dbl_pass.m` step by step in Python. It keeps the same physical structure as the MATLAB script:

1. Build the time and frequency grids.
2. Define the AWG modulation-frequency grid.
3. Choose 11 AWG tones from 95 to 105 MHz.
4. Sample random cosine and sine coefficients and normalize their total power to `Pmax`.
5. Construct the time-domain modulation signal `AM(t)`.
6. Simulate the double-pass response with `RF(t) = AM(t)^2`.
7. Compute and plot the spectra of `AM(t)` and `RF(t)`.

## Imports And Plot Style

The MATLAB script uses built-in plotting and FFT functions. Here we use NumPy for numerical work and Matplotlib for the figures.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True})

## Time And Frequency Axes

This reproduces the first parameter block of the MATLAB script. Time is measured in microseconds and frequency in MHz, so `fs = 1 / dt` is also in MHz.

In [ ]:
dt = 1e-3          # MATLAB: dt=1e-3; sampling interval in us
L = int(1e5)      # MATLAB: L=1e5; number of sampling points

t = np.arange(L) * dt       # MATLAB: t=(0:(L-1))*dt
fs = 1 / dt                 # MATLAB: fs=1/dt; sampling frequency
df = fs / L                 # MATLAB: df=fs/L; frequency resolution
f = np.arange(L // 2) * df  # MATLAB: f=(0:L/2-1)*df

print(f'fs = {fs:g} MHz')
print(f'df = {df:g} MHz')
print(f'time points = {len(t)}')
print(f'positive FFT bins = {len(f)}')

## AWG Modulation-Frequency Grid

The MATLAB script defines a frequency grid centered at 100 MHz with 0.1 MHz spacing. The coefficient arrays `an` and `bn` store cosine and sine amplitudes on that grid.

In [ ]:
f0 = 100.0       # MATLAB: f0=100; center frequency in MHz
dfm = 0.1        # MATLAB: dfm=0.1; modulation-frequency spacing in MHz
Nm = 200         # MATLAB: Nm=200; number of modulation-frequency intervals

fm = f0 + np.arange(-Nm // 2, Nm // 2 + 1) * dfm  # MATLAB: fm=f0+(-Nm/2:Nm/2)*dfm
an = np.zeros_like(fm)  # MATLAB: an=zeros(size(fm)); cosine amplitudes
bn = np.zeros_like(fm)  # MATLAB: bn=zeros(size(fm)); sine amplitudes

print(f'fm range: {fm[0]:.1f} to {fm[-1]:.1f} MHz')
print(f'number of modulation frequencies: {len(fm)}')

## Select Nonzero AWG Tones

The active AWG tones are the integer frequencies from 95 to 105 MHz. MATLAB uses one-based indices; Python uses zero-based indices, so the index formula is shifted by one.

In [ ]:
freq = np.arange(95.0, 106.0, 1.0)  # MATLAB: freq=95:105; 11 frequencies

# MATLAB: vec=1+Nm/2+(freq-f0)/dfm;
# Python equivalent without the one-based '+1'.
vec = (Nm // 2 + np.rint((freq - f0) / dfm)).astype(int)

print('selected AWG frequencies:', freq)
print('selected Python indices:', vec)
print('fm[vec]:', fm[vec])

## Random Coefficients And Power Normalization

The MATLAB script samples random coefficients with `rand(size(freq))-0.5` and then normalizes the total input power to `Pmax = 1`.

Set `SEED = None` if you want a fresh random draw each time, closer to the MATLAB script without `rng(...)`.

In [ ]:
SEED = 42         # Reproducible Python run; set to None for non-deterministic random coefficients.
Pmax = 1.0        # MATLAB: Pmax=1; target total input power

rng = np.random.default_rng(SEED)
an[vec] = rng.random(len(freq)) - 0.5  # MATLAB: an(vec)=rand(size(freq))-0.5
bn[vec] = rng.random(len(freq)) - 0.5  # MATLAB: bn(vec)=rand(size(freq))-0.5

pwr = an @ an + bn @ bn                # MATLAB: pwr=an*an'+bn*bn'
an = an / np.sqrt(pwr / Pmax)          # MATLAB: an=an/(pwr/Pmax)^0.5
bn = bn / np.sqrt(pwr / Pmax)          # MATLAB: bn=bn/(pwr/Pmax)^0.5
powerin = an @ an + bn @ bn            # MATLAB: powerin=an*an'+bn*bn'

print(f'powerin = {powerin:.12f}')
print('nonzero an:', np.round(an[vec], 4))
print('nonzero bn:', np.round(bn[vec], 4))

## Build The Time-Domain AWG Signal

This cell reproduces the MATLAB loop:

`AM = AM + an(nf)*cos(2*pi*fm(nf)*t) - bn(nf)*sin(2*pi*fm(nf)*t)`

In [ ]:
AM = np.zeros_like(t)
for nf in vec:
    AM += an[nf] * np.cos(2 * np.pi * fm[nf] * t) - bn[nf] * np.sin(2 * np.pi * fm[nf] * t)

print(f'AM min/max: {AM.min():.4f}, {AM.max():.4f}')

## Double-Pass Response

The revised MATLAB model simulates the double-pass AOM/RF response with a square-law nonlinearity.

In [ ]:
RF = AM ** 2  # MATLAB: RF=AM.^2

print(f'RF min/max: {RF.min():.4f}, {RF.max():.4f}')

## Time-Domain Plots

These correspond to MATLAB `figure(1); plot(t,AM)` and `figure(2); plot(t,RF)`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)

axes[0].plot(t, AM)
axes[0].set_title('AWG modulation signal AM(t)')
axes[0].set_ylabel('AM')

axes[1].plot(t, RF)
axes[1].set_title('Double-pass response RF(t)=AM(t)^2')
axes[1].set_xlabel('time (us)')
axes[1].set_ylabel('RF')

fig.tight_layout()
plt.show()

## Spectrum Of AM(t)

This reproduces the MATLAB block that computes `spekAM`, splits it into real and imaginary parts, and plots the 75 to 125 MHz window.

In [ ]:
spekAM = np.fft.fft(AM) / (L / 2)       # MATLAB: spekAM=fft(AM)/(L/2)
powerAM = np.sum(np.abs(spekAM[:L // 2]) ** 2)
aspekAM = spekAM[:L // 2].real
bspekAM = spekAM[:L // 2].imag

print(f'powerAM = {powerAM:.12f}')

maxab = max(np.max(np.abs(aspekAM)), np.max(np.abs(bspekAM)))
plt.figure(figsize=(9, 4))
plt.plot(f, aspekAM, '-o', markersize=2, label='real')
plt.plot(f, bspekAM, '-o', markersize=2, label='imag')
plt.xlim(75, 125)
plt.ylim(-1.1 * maxab, 1.1 * maxab)
plt.title('Spectrum of AM(t)')
plt.xlabel('frequency (MHz)')
plt.ylabel('complex amplitude')
plt.legend()
plt.show()

## Spectrum Of RF(t)

This reproduces the final MATLAB block. The plotted window is 175 to 225 MHz, matching the double-pass spectrum region in the script.

In [ ]:
spekRF = np.fft.fft(RF) / (L / 2)       # MATLAB: spekRF=fft(RF)/(L/2)
powerRF = np.sum(np.abs(spekRF[:L // 2]) ** 2) / 2
aspekRF = spekRF[:L // 2].real
bspekRF = spekRF[:L // 2].imag

print(f'powerRF = {powerRF:.12f}')

# MATLAB uses max(abs(aspekRF(50:end))) and max(abs(bspekRF(50:end))).
# Python index 49 corresponds to MATLAB index 50.
maxab = max(np.max(np.abs(aspekRF[49:])), np.max(np.abs(bspekRF[49:])))
plt.figure(figsize=(9, 4))
plt.plot(f, aspekRF, '-o', markersize=2, label='real')
plt.plot(f, bspekRF, '-o', markersize=2, label='imag')
plt.xlim(175, 225)
plt.ylim(-1.1 * maxab, 1.1 * maxab)
plt.title('Spectrum of RF(t)')
plt.xlabel('frequency (MHz)')
plt.ylabel('complex amplitude')
plt.legend()
plt.show()